# ASSIGNMENT NLP – 3  
## Chatbot using Hugging Face Transformers

### Objective
Build a console-based chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

### Model Used
- microsoft/Phi-3-mini-4k-instruct

### Tools and Technologies
- Python
- Hugging Face Transformers
- PyTorch
- Jupyter Notebook / Google Colab

In [ ]:
!pip install -q transformers accelerate torch sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print("Loading tokenizer and model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully.")

Loading tokenizer and model...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully.


## Chatbot Workflow

The chatbot follows these steps:

1. Accept user input from the console.
2. Build a proper chat-format prompt.
3. Add limited conversation history.
4. Generate a response using the transformer model.
5. Display the answer.
6. Continue until the user types `exit` or `quit`.

This chatbot uses a pure transformer-based approach without a separate knowledge base.

In [ ]:
conversation_history = []
MAX_TURNS = 3

In [ ]:
def build_messages(user_input, history):
    system_message = (
        "You are a helpful, accurate, and polite AI assistant. "
        "Answer clearly and naturally. "
        "For greetings, respond with a friendly greeting. "
        "For factual questions, give short and correct answers. "
        "Do not invent personal details such as your own human name. "
        "If you do not know the answer, say you are not sure."
    )

    messages = [{"role": "system", "content": system_message}]

    for user_msg, bot_msg in history[-MAX_TURNS:]:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": user_input})
    return messages

In [ ]:
def generate_response(user_input, history):
    messages = build_messages(user_input, history)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return response

In [ ]:
def chatbot():
    print("Chatbot: Hello! I am your AI assistant.")
    print("Type 'exit' or 'quit' to end the chat.\n")

    while True:
        user_input = input("User: ").strip()

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        if not user_input:
            print("Chatbot: Please enter a message.\n")
            continue

        response = generate_response(user_input, conversation_history)

        conversation_history.append((user_input, response))

        print(f"Chatbot: {response}\n")

In [ ]:
chatbot()

Chatbot: Hello! I am your AI assistant.
Type 'exit' or 'quit' to end the chat.

User: Hi


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Chatbot: Hello! How can I assist you today?

User: How are you?
Chatbot: I'm doing well, thank you for asking! How about you?

User: I am great,who are you?
Chatbot: I'm an AI here to help you with any questions or tasks you might have.

User: How can you help me?
Chatbot: I can assist you with a wide range of tasks, from providing information and answering questions to helping you organize your schedule and manage your tasks.

User: Ok,so can you tell that in yesterday match who won between RR and CSK?
Chatbot: I'm sorry, but I don't have real-time access to current events or sports results. I recommend checking a reliable sports news website or app for the most up-to-date information on the match between Rajasthan Royals (RR) and Chennai Super Kings (CSK).

User: Till what details you can say?
Chatbot: I can provide information up to my last training data, which was in September 2021. For the most recent events or developments, it's best to consult current sources.

User: Ok
Chatbot:

In [ ]:
print("\nConversation History:\n")

for i, (user_msg, bot_msg) in enumerate(conversation_history, 1):
    print(f"Turn {i}")
    print(f"User: {user_msg}")
    print(f"Chatbot: {bot_msg}")
    print("-" * 40)


Conversation History:

Turn 1
User: Hi
Chatbot: Hello! How can I assist you today?
----------------------------------------
Turn 2
User: How are you?
Chatbot: I'm doing well, thank you for asking! How about you?
----------------------------------------
Turn 3
User: I am great,who are you?
Chatbot: I'm an AI here to help you with any questions or tasks you might have.
----------------------------------------
Turn 4
User: How can you help me?
Chatbot: I can assist you with a wide range of tasks, from providing information and answering questions to helping you organize your schedule and manage your tasks.
----------------------------------------
Turn 5
User: Ok,so can you tell that in yesterday match who won between RR and CSK?
Chatbot: I'm sorry, but I don't have real-time access to current events or sports results. I recommend checking a reliable sports news website or app for the most up-to-date information on the match between Rajasthan Royals (RR) and Chennai Super Kings (CSK).
---

## Notes

This chatbot uses a pure transformer-based approach without a separate knowledge base.  
The model is instruction-tuned and prompted in chat format to improve the quality of responses.  
Low-randomness decoding is used to make answers more stable and direct.